# Laboratorio 6 — Regresión Lineal y Logística
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701
**Fecha:** Miércoles 23 de septiembre de 2026 · Sesión 6 de 17
**Bloques:** 6-8 (11:40-13:40, 105 min efectivos + recreo) · Lab. Central 114
**Unidad 2:** Algoritmos supervisados y no supervisados (inicio)

**Objetivo:** entrenar los dos modelos de la mañana con datos reales, leer sus coeficientes, y ver
la frontera de decisión de la regresión logística dibujada de verdad. El pipeline de la Unidad 1
(split, estratificación, escalamiento, validación cruzada) ya no se explica: se aplica.

> **De dónde venimos:** en la Unidad 1 llamamos `fit()` decenas de veces sin preguntarnos qué
> pasaba adentro. Hoy abrimos la caja: `fit()` busca pesos $\mathbf{w}$ que minimizan una función
> de costo — MSE en la regresión lineal, log-loss en la logística — y esos pesos se pueden leer.

---
## Parte 1 · Regresión lineal sobre Diabetes

### 1. Setup y primer ajuste

Diabetes: 442 pacientes, 10 features clínicas (edad, sexo, IMC, presión, 6 mediciones de sangre
`s1`-`s6`) y un target **continuo**: la progresión de la enfermedad un año después.

La carga y el split ya están dados (hábito de la Clase 5: split antes de tocar nada).

**Tu tarea:** crea un `LinearRegression()` llamado `reg` y entrénalo con el train.

In [1]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

diabetes = load_diabetes(as_frame=True)
X, y = diabetes.data, diabetes.target   # y: progresión de la enfermedad a un año, valor continuo

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

### 2. Leer los coeficientes

**Antes de correr nada, responde:** ¿qué feature esperas que pese más en la progresión de la
enfermedad?

> *(tu respuesta)*

**Tu tarea:**
1. Arma una `pd.Series` con `reg.coef_` indexada por `X.columns` y ordénala (`.sort_values()`).
2. Imprime también `reg.intercept_`.

In [2]:
import pandas as pd

**Responde aquí:**
1. Elige dos coeficientes y léelos: ¿el signo dice que la progresión aumenta o disminuye con esa
   feature? ¿Cuál pesa más?
2. Las features de este dataset vienen **escaladas de fábrica**. ¿Por qué eso es lo que permite
   comparar las magnitudes entre ellas? *(gancho a la Clase 3)*
3. `s1` (colesterol total) y `s2` (LDL) tienen coeficientes grandes y de **signo opuesto**. Si
   ambas miden casi lo mismo, ¿tiene sentido decir que "más colesterol reduce la progresión"?

> *(tus respuestas)*

### 3. Evaluación: MSE y R²

**Tu tarea:**
1. Predice sobre `X_test` y guarda el resultado en `y_pred`.
2. Imprime `mean_squared_error` y `r2_score`.
3. Grafica valor real (eje x) vs predicho (eje y), con la diagonal $y = x$ en rojo punteado.

In [3]:
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

**Responde aquí:**
1. Explica tu $R^2$ en una frase: ¿qué proporción de la variabilidad de la progresión explica el
   modelo? (recuerda: 1.0 = perfecto, 0 = tan bueno como predecir siempre el promedio)
2. El MSE está en unidades **al cuadrado**. Sácale raíz: ¿cuántos puntos de progresión se
   equivoca el modelo "en promedio", si el target va de 25 a 346?
3. En el gráfico, ¿los puntos se pegan a la diagonal? ¿En qué zona (valores bajos o altos) se
   equivoca más?

> *(tus respuestas)*

---
## ☕ Recreo

---
## Parte 2 · Regresión logística sobre Breast Cancer

### 4. Setup y ajuste

Breast Cancer: 569 tumores, 30 features calculadas de imágenes de biopsia, target binario
(**0 = maligno, 1 = benigno**). Es el mismo dataset que quedó como ejercicio propuesto en la
Clase 4.

**Tu tarea:**
1. Split con `test_size=0.3, random_state=42, stratify=y_c` → `X_train_c, X_test_c, y_train_c, y_test_c`.
2. Escala con `StandardScaler` (`fit` en train, `transform` en test) → `X_train_c_s`, `X_test_c_s`.
3. Entrena `logreg = LogisticRegression()` e imprime su accuracy en test.

In [4]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

cancer = load_breast_cancer(as_frame=True)
X_c, y_c = cancer.data, cancer.target   # 0 = maligno, 1 = benigno

**Responde aquí:** en esta celda aplicaste dos hábitos de la Unidad 1 sin que nadie los pidiera
explícitamente en el código de ejemplo de la teoría. ¿Cuáles, y de qué clase viene cada uno?

> *(tu respuesta)*

### 5. Predicción de probabilidades

**Tu tarea:** para las primeras 5 muestras del test, imprime `logreg.predict_proba(...)`,
`logreg.predict(...)` y las etiquetas reales `y_test_c.values[:5]`.

In [5]:
import numpy as np

**Responde aquí:**
1. Lee una fila de `predict_proba` en voz alta: ¿qué es cada columna? ¿Cuánto suman?
2. ¿Cómo se obtiene `predict` a partir de `predict_proba`?
3. Busca la fila **menos segura** de las 5. Si fueras médico, ¿tratarías igual un 0.99 de
   "maligno" que un 0.71?
4. KNN también podía dar una "probabilidad". ¿De dónde salía allá, y de dónde sale acá?

> *(tus respuestas)*

### 6. Frontera de decisión en 2D

Con 30 features no se puede dibujar nada. Nos quedamos con **dos** (`worst radius`,
`worst texture`) para ver la frontera con nuestros propios ojos.

**Tu tarea:**
1. Split estratificado de `X2` (mismos parámetros de siempre) y escalamiento → `X2_train_s`.
2. Entrena `logreg2 = LogisticRegression()`.
3. Completa `Z` (la predicción sobre cada punto de la grilla) y descomenta el gráfico.

In [6]:
# Dos features para poder graficar en 2D
cols = ["worst radius", "worst texture"]
X2 = cancer.data[cols]

# 1. Split estratificado (test_size=0.3, random_state=42) y escalamiento de X2 → X2_train_s
# 2. Entrena logreg2 = LogisticRegression() sobre X2_train_s


# 3. La grilla y el gráfico ya están armados: descomenta y completa Z
# xx, yy = np.meshgrid(
#     np.linspace(X2_train_s[:, 0].min() - 1, X2_train_s[:, 0].max() + 1, 200),
#     np.linspace(X2_train_s[:, 1].min() - 1, X2_train_s[:, 1].max() + 1, 200),
# )
# Z = ...   # predice cada punto de la grilla: np.c_[xx.ravel(), yy.ravel()] → .reshape(xx.shape)
#
# plt.contourf(xx, yy, Z, alpha=0.3)
# plt.scatter(X2_train_s[:, 0], X2_train_s[:, 1], c=y2_train, edgecolor="k")
# plt.xlabel(cols[0] + " (escalada)")
# plt.ylabel(cols[1] + " (escalada)")
# plt.show()

**Responde aquí:**
1. ¿Qué forma tiene la frontera entre las dos regiones de color? Relaciónalo con la ecuación
   $\mathbf{w}\cdot\mathbf{x} + w_0 = 0$ de la teoría.
2. Imprime `logreg2.coef_`: ¿cuál de las dos features pesa más? ¿Se nota en la inclinación de la
   frontera?
3. ¿Hay puntos del lado "equivocado"? ¿Podría una recta separarlos todos?

> *(tus respuestas)*

---
## Parte 3 · Comparación y cierre

### 7. Logística vs KNN, cara a cara

**Tu tarea:**
1. Arma dos pipelines: `StandardScaler` + `KNeighborsClassifier()` y `StandardScaler` +
   `LogisticRegression()`.
2. Compara ambos con `cross_val_score(..., X_c, y_c, cv=5)` — media ± desviación.

In [7]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

**Responde aquí:** si los scores son parecidos, ¿qué modelo elegirías para un hospital, y por qué?
¿Qué tiene la logística que KNN no tiene? Conecta con la Clase 2: ¿cuál es paramétrico y cuál no?

> *(tu respuesta)*

### 8. Ejercicio propuesto (no evaluado)

Repite la regresión logística sobre **Wine** (3 clases): pipeline escalador + `LogisticRegression`,
CV de 5 folds, y un split para mirar `coef_` y `predict_proba`.

¿Qué forma tiene ahora `coef_`? ¿Cuántas columnas tiene `predict_proba` y cuánto suma cada fila?
`sklearn` resolvió el caso multiclase solo — con **softmax**, la generalización de la sigmoide a
más de dos clases que se mencionó en la teoría.

In [8]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
X_w, y_w = wine.data, wine.target

---
**Moraleja del lab:** hoy se abrió por primera vez la caja negra de `fit()` — no es magia, son
pesos que se ajustan minimizando una función de costo concreta, y esos pesos se pueden leer.

**Próxima clase:** árboles de decisión y bosques aleatorios — el árbol vuelve, esta vez con la
maquinaria formal detrás de por qué el del Lab 1 llegó a score 1.0.